# Libraries

In [1]:
#!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install "lm_eval[hf]"
!pip -q install --upgrade transformers
!pip -q install langdetect --break-system-packages #for If-eval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 76.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 81.3 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.0 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done


In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
MODEL_NAME       = "Llama-3.2-3B-Instruct-SparseGPT-50"
MODEL_PRETRAINED = "EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT"
SUB_FOLDER = "Models/50"
SEED             = 42

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Evaluations

**Configurations**

In [ ]:
# Task definition
TASKS = [
    # Math
     Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=12,  max_gen_toks=1024),

    # Reasoning
     Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=16),

    # Instruction Following
     Task("ifeval",       "ifeval",                    "instruction_following",batch_size=14,  max_gen_toks=1024),

    # Perplexity
     Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Knowledge
     Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),
]


# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    f"subfolder={SUB_FOLDER}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    #"enable_thinking=False",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [9]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "WARNING",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)

    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[MATH] gsm8k


2026-04-06:14:22:26 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-06:14:22:34 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-04-06:14:22:36 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 219849.27 examples/s]
2026-04-06:14:23:29 WARNING  [evaluator:333] Overwriting default num_fewshot of gsm8k_cot from 8 to 8
2026-04-06:14:23:29 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 1319/1319 [2:42:06<00:00,  7.37s/it]  
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/50', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 8, batch_size: 6
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|---------|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     8|exact_match|↑  |0.2282|±  |0.0116|
|         |       |strict-match    |     8|exact_match|↑  |0.2161|±  |0.0113|

Done

[INSTRUCTION_FOLLOWING] ifeval


2026-04-06:17:06:09 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-06:17:06:15 INFO     [_cli.run:376] Selected Tasks: ['ifeval']
2026-04-06:17:06:17 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Generating train split: 100%|██████████| 541/541 [00:00<00:00, 47589.57 examples/s]
2026-04-06:17:06:30 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 541/541 [1:38:10<00:00, 10.89s/it]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/50', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 0, batch_size: 6
|Tasks |Version|Filter|n-shot|        Metric         |   |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|---|-----:|---|------|
|ifeval|      4|none  |     0|inst_level_loose_acc   |↑  |0.5887|±  |   N/A|
|      |       |none  |     0|inst_level_strict_acc  |↑  |0.5504|±  |   N/A|
|      |       |none  |     0|prompt_level_loose_acc |↑  |0.4695|±  |0.0215|
|      |       |none  |     0|prompt_level_strict_acc|↑  |0.4288|±  |0.0213|

Done

[REASONING] arc_challenge


...

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/50', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 8
|    Tasks    |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-------------|------:|------|-----:|--------|---|-----:|---|-----:|
|arc_challenge|      1|none  |     0|acc     |↑  |0.3157|±  |0.0136|
|             |       |none  |     0|acc_norm|↑  |0.3601|±  |0.0140|

Done

[PERPLEXITY] wikitext


2026-04-06:18:47:36 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
2026-04-06:18:47:36 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
Loading weights: 100%|██████████| 254/254 [00:01<00:00, 142.18it/s]
2026-04-06:18:47:49 WARNING  [api.task:729] [Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-04-06:18:47:49 WARNING  [api.task:741] [Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-04-06:18:47:49 WARNING  [api.task:729] [Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-04-06:18:47:49 WARNING  [api.task:741] [Task: wikitext] metric byte_perplexity is defined, but h

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/50', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: None, num_fewshot: 0, batch_size: 1
| Tasks  |Version|Filter|n-shot|    Metric     |   | Value |   |Stderr|
|--------|------:|------|-----:|---------------|---|------:|---|------|
|wikitext|      2|none  |     0|bits_per_byte  |↓  | 0.8775|±  |   N/A|
|        |       |none  |     0|byte_perplexity|↓  | 1.8372|±  |   N/A|
|        |       |none  |     0|word_perplexity|↓  |25.8561|±  |   N/A|

Done

[KNOWLEDGE] mmlu


2026-04-06:18:54:20 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-04-06:18:54:20 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-06:18:54:26 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 2006.65 examples/s]
2026-04-06:18:56:01 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_abstract_algebra from None to 5
2026-04-06:18:56:01 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_anatomy from None to 5
2026-04-06:18:56:01 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_astronomy from None to 5
2026-04-06:18:56:01 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_biology from None to 5
2026-04-06:18:56:01 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_chemistry from None to 5
2026-04-06:18:56:01 WARNING  [evaluato

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/50', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({}), limit: 1400.0, num_fewshot: 5, batch_size: 2
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.4351|±  |0.0041|
| - humanities                          |      2|none  |      |acc   |↑  |0.3872|±  |0.0071|
|  - formal_logic                       |      1|none  |     5|acc   |↑  |0.2302|±  |0.0376|
|  - high_school_european_history       |      1|none  |     5|acc   |↑  |0.4848|±  |0.0390|
|  - high_school_us_history             |      1|none  |     5|acc   |↑  |0.4559|±  |0.0350|
|  - high_school_world_history          |      1|none  |     5|acc   |↑  |0.5359|±  |0.0325|
|  - international_law             

# Save reports

In [10]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)